# Customer Shopping Behavior Analysis\nEnd-to-end notebook covering data cleaning, EDA, and customer segmentation.

In [ ]:
import pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nfrom pathlib import Path\n\nsns.set_theme(style='whitegrid')

In [ ]:
data_path = Path('../data/customer_shopping_data.csv')\ndf = pd.read_csv(data_path)\ndf.head()

## Data Cleaning and Preprocessing (Pandas)

In [ ]:
df['age'] = df['age'].fillna(df['age'].median())\ndf['order_date'] = pd.to_datetime(df['order_date'])\ndf['discount_pct'] = df['discount_pct'].clip(lower=0, upper=80)\ndf['quantity'] = df['quantity'].clip(lower=1)\ndf['unit_price'] = df['unit_price'].clip(lower=0)\ndf['total_amount'] = df['quantity'] * df['unit_price'] * (1 - df['discount_pct'] / 100)\ndf.isna().sum()

## Exploratory Data Analysis (EDA)

In [ ]:
city_sales = df.groupby('city', as_index=False)['total_amount'].sum().sort_values('total_amount', ascending=False)\ncategory_sales = df.groupby('product_category', as_index=False)['total_amount'].sum().sort_values('total_amount', ascending=False)\ndisplay(city_sales.head())\ndisplay(category_sales)

In [ ]:
plt.figure(figsize=(8,4))\nsns.barplot(data=category_sales, x='product_category', y='total_amount', palette='Blues_d')\nplt.xticks(rotation=45, ha='right')\nplt.title('Sales by Product Category')\nplt.tight_layout()

## Customer Segmentation (RFM-style scoring)

In [ ]:
snapshot_date = df['order_date'].max() + pd.Timedelta(days=1)\nrfm = df.groupby('customer_id').agg(\n    recency=('order_date', lambda x: (snapshot_date - x.max()).days),\n    frequency=('order_id', 'count'),\n    monetary=('total_amount', 'sum')\n).reset_index()\nrfm['r_score'] = pd.qcut(rfm['recency'], 4, labels=[4,3,2,1])\nrfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 4, labels=[1,2,3,4])\nrfm['m_score'] = pd.qcut(rfm['monetary'], 4, labels=[1,2,3,4])\nrfm['segment_score'] = rfm[['r_score','f_score','m_score']].astype(int).sum(axis=1)\nrfm['segment'] = pd.cut(rfm['segment_score'], bins=[0,5,8,12], labels=['At Risk','Potential Loyalist','Champion'])\nrfm.sort_values('segment_score', ascending=False).head()

In [ ]:
output_path = Path('../data/customer_shopping_data_cleaned.csv')\ndf.to_csv(output_path, index=False)\nprint(f'Cleaned dataset exported to {output_path.resolve()}')